In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================

# 소비기한 추출 파이프라인 (2-Pass Hybrid)

```
원본 → ① EXIF 보정 → ② 1패스: 축소본 전체 OCR → 날짜 후보 + 위치
                     → ③ 2패스: 후보 영역만 원본 해상도 크롭 → 재인식
                     → ④ 선택: 가장 늦은 날짜 (= 소비기한, 운영진 확정 규칙)
                     → ⑤ 후보 없음 → 사전확률 날짜 (부분점수 정책)
```

- 비싼 고해상도 인식을 전체가 아닌 **후보 영역에만** 쓴다. 1패스는 리콜, 2패스는 정밀도.
- 9자리 이상 연속 숫자(품목보고번호·바코드)는 파싱 전에 마스킹한다.
- 모든 가중치는 `./weights` 에서 오프라인 로드한다 (`download_weights.sh` 선실행).

In [ ]:
import re, glob, time
from collections import Counter

import cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
import easyocr

# torch 기본값은 물리 코어 수. 논리 코어까지 쓰면 검출기가 ~25% 빨라진다 (실측: 4→8 스레드 12.3s→8.9s).
torch.set_num_threads(int(os.environ.get("ITDA_THREADS", 0)) or os.cpu_count() or 4)   # 개발용: ITDA_THREADS 로 스레드 제한해 느린 채점 서버를 흉내 냄. 채점 환경엔 미설정

# ----- 파이프라인 파라미터 -----
USE_GPU        = False
WEIGHTS_DIR    = "./weights"

# 1패스 해상도 사다리(긴 변 px). 앞 단계에서 후보가 나오면 멈추고, 없을 때만 다음 단계로 올라간다.
# 검출기(CRAFT) 비용은 픽셀 수에 비례 — 실측(4스레드): 480→1.8s, 640→2.9s, 800→4.6s, 1024→7.8s, 1280→12.2s.
# 640 에서는 영양성분표 잔글씨가 검출되지 않아 잡음이 줄고 큼직한 날짜 스탬프는 잡히지만,
# 고해상도 사진에 작게 인쇄된 날짜(3024x4032 의 라벨 등)는 놓친다 → 그런 장만 1024 로 재시도.
# 채점 예산 4.8s/장(500장/2400s). 대부분 640 에서 끝나 평균은 낮고, 어려운 장에만 예산을 더 쓴다.
PASS1_LADDER   = (640, 1024)

# 시간 예산 가드. 채점 서버 속도를 모르므로(개발 PC 2스레드 실측: 500장 환산 2438s 로 턱걸이 초과), 실행 중 진행 속도로 총 소요를 예측해
# TIME_BUDGET_S 를 넘길 것 같으면 남은 장은 경량 모드(사다리 640 만, 영문 폴백 생략)로 전환한다. 타임아웃(2400s) 시 정량 전체 0점을 구조적으로 막는다.
TIME_BUDGET_S  = int(os.environ.get("ITDA_TIME_BUDGET", 2000))   # 채점 제한 2400s 에서 모델 로드·저장 여유를 뺀 값. 개발용 env 로 낮춰 발동 테스트 가능
# 경량 단계 (실행 중 자동 상승). Colab 2vCPU 실측 장당 14.2s(500장 118분)처럼 느린 서버에서도 완주하도록 3단계로 둔다.
#   0 정상  : 사다리 640→1024, 영문 폴백, 2패스, 박스 30            (개발 PC 8스레드 4.1s/장)
#   1 경량  : 사다리 640 만, 폴백 생략                              (~75%)
#   2 최소  : 사다리 640 만, 박스 10, 2패스 생략                    (~85%)  ※ 480px 로 줄이면 정확도 53→6% (실측) — 해상도는 못 줄임
#   3 비상  : 남은 장은 NONE — 어떤 경우에도 submission.csv 는 생성  (~0)
# 검출기(CRAFT) 비용이 장당 시간의 절반 이상이라 2단계의 절약 폭은 작다. Colab 급으로 느린 서버에선 3단계가 불가피 — 완주와 CSV 를 지키는 장치.
FAST_LEVEL     = int(os.environ.get("ITDA_FAST_LEVEL", 0))   # 개발용 env 로 특정 단계를 강제해 정확도 손실을 측정. 채점 환경엔 미설정

# 1패스 인식 예산. 인식기는 박스당 ~0.17s(실측)라 글자 많은 라벨(50박스=8.5s)에서 검출기보다 비싸다.
# 박스를 글자 높이 내림차순으로 인식하다가 연도 포함 날짜를 찾으면, 그 높이의 EARLY_STOP_RATIO 배 미만인 박스는 건너뛴다.
# (날짜 스탬프는 영양성분표 잔글씨보다 크고, 제조일자·소비기한 짝은 보통 같은 글꼴 크기.)
MAX_BOXES        = 30          # 장당 인식 박스 상한 (날짜 비슷한 텍스트를 하나라도 봤을 때)
MAX_BOXES_HARD   = 30          # 날짜 비슷한 것도 못 봤을 때의 상한. 50 으로 올려 봤으나(v4) 완전일치 +0.2%p 에 시간 +17% → 채택 안 함. 30 = 소프트 상한 비활성
EARLY_STOP_RATIO = 0.6

# 2패스용 원본 상한(긴 변 px). 이보다 큰 JPEG 은 디코딩 단계(draft)에서 축소한다.
# 24MP 원본을 그대로 풀면 디코딩+EXIF 회전만 2~3초. 날짜 스탬프 재인식에는 2000px 이면 충분하다.
LOAD_MAX_LONG  = 2000
PASS2_MARGIN   = 0.30          # 2패스 크롭 여백 비율 (박스 크기 대비). 640px 검출 박스를 원본 좌표로 환산하면 오차가 커서 가장자리 숫자가 잘림 → 넉넉히

# 연도 허용 범위. 수집 시점(~2026) 기준으로 이미 만료된 구형 사진(2019~)부터 장기보관 식품(~2031)까지.
# 범위를 넓히면 '33 01 15' 같은 잡음이 2033년으로 통과해 '가장 늦은 날짜' 규칙을 오염시킨다.
YEAR_MIN, YEAR_MAX = 2017, 2031   # 라벨 653장 실측: 2017~2030 (2017 은 3건). 범위 밖은 LOT·전화번호 등 잡음
YY_SOFT_MAX = 2027             # 2자리 연도가 모호할 때(YY.MM.DD 와 DD.MM.YY 둘 다 성립) 이 해를 넘는 YY 해석은 버림. '28/02/22' → 2022-02-28. 라벨 중 2028+ 는 4건

DEBUG = os.environ.get("ITDA_DEBUG", "") == "1"   # 개발용: 장별 OCR 라인·후보·소요시간 출력. 채점 환경에선 미설정.
DEBUG_CSV = os.environ.get("ITDA_DEBUG_CSV", "")   # 개발용: 장별 선택값·신뢰도·후보를 CSV 로 저장 (정확도·정밀도-커버리지 분석용). 채점 환경에선 미설정.
MASK_DIGITS_GE = 9             # 이 길이 이상 연속 숫자는 날짜가 아님 (품목보고번호 11~14자리, 바코드 13자리)

ALLOW_P1 = "0123456789./-: "   # 1패스 인식 allowlist (시각 구분자 ':' 포함해 시간을 시간으로 읽게 함)
ALLOW_P2 = "0123456789./- "    # 2패스 인식 allowlist
# 영문 월 이름 폴백 (BBE NOV 29 2021 — 라벨 3.7%). 숫자 allowlist 에선 월이 사라지므로, 후보가 0개일 때만
# 큰 박스 몇 개를 영문 포함 allowlist 로 재인식한다. 평소엔 영문을 안 넣는 이유: O/0, I/1, S/5 혼동으로 숫자 정확도가 떨어짐.
ALLOW_MON = "0123456789./-: ABCDEFGHIJKLMNOPQRSTUVWXYZ"
MON_FALLBACK_BOXES = 12        # 재인식 박스 수 (큰 글자 순) → 최대 ~2초

# 후보 0개(OCR 전면 실패)일 때 정책.
#  - "none" : NONE,NONE,NONE. 스펙 준수 기본값. 운영진 확정: NONE 은 정답 라벨 값으로 실제 존재한다(예: 연도 없는 우유 → NONE-10-14).
#  - "prior": 사전확률 날짜. 라벨이 실제 날짜인데 OCR 이 통째로 놓친 경우엔 기대값이 높다. 검증셋으로 비교 후 결정.
NONE_POLICY = "none"           # "none" | "prior"
PRIOR_DATE  = (2027, 1, 1)     # TODO: 검증셋 라벨 분포로 갱신

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

In [ ]:
def load_image(path, max_long=LOAD_MAX_LONG):
    """EXIF Orientation 보정 후 BGR ndarray 반환.
    - 데이터셋의 8.7%가 Orientation=6 (시계 90도). cv2.imread 는 EXIF 를 무시하므로 PIL 로 연다.
    - MPO(.jpg 확장자의 다중프레임), PNG-as-jpg 등 포맷 불일치도 PIL 이 흡수한다.
    - JPEG 은 draft 모드로 디코딩 단계에서 1/2·1/4·1/8 축소한다. 요청 크기 이상은 보장되므로 max_long 은 하한이다.
    - EasyOCR 은 cv2 관행(BGR) 입력을 가정하므로 채널 순서를 맞춘다."""
    with Image.open(path) as im:
        if max_long and im.format in ("JPEG", "MPO"):
            w, h = im.size
            s = max_long / max(w, h)
            if s < 1.0:
                im.draft("RGB", (max(1, int(w * s)), max(1, int(h * s))))
        im = ImageOps.exif_transpose(im).convert("RGB")
        rgb = np.array(im)
    return cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)


def resize_long(img, long_side):
    """긴 변을 long_side 로 축소. 확대는 하지 않는다. (결과, 축척) 반환."""
    h, w = img.shape[:2]
    s = long_side / max(h, w)
    if s >= 1.0:
        return img, 1.0
    return cv2.resize(img, (round(w * s), round(h * s)), interpolation=cv2.INTER_AREA), s


def crop_with_margin(img, bbox, margin):
    """bbox=(x0,y0,x1,y1) 원본 좌표. 박스 크기 비율만큼 여백을 두고 크롭."""
    H, W = img.shape[:2]
    x0, y0, x1, y1 = bbox
    mw, mh = (x1 - x0) * margin, (y1 - y0) * margin
    x0, x1 = int(max(0, x0 - mw)), int(min(W, x1 + mw))
    y0, y1 = int(max(0, y0 - mh)), int(min(H, y1 + mh))
    if x1 <= x0 or y1 <= y0:
        return None
    return img[y0:y1, x0:x1]

In [ ]:
# 날짜 후보 정규식. (?<!\d) / (?!\d) 로 더 긴 숫자열의 일부를 잘라 읽는 것을 막는다.
_SEP = r"\s*[.\-/]\s*"
# 앞에 잡음 숫자 하나가 붙어도 허용 ('42025.10.09' — 도트 프린터 주변 얼룩이 숫자로 읽힘). 뒤 잡음은 greedy 2자리로 흡수.
_P_YYYY = re.compile(r"(?<!\d)\d?(20\d{2})[.\-/\s]+(\d{1,2})[.\-/\s]+(\d{1,2})")   # 2027.02.14 / 2027 02 14 / '2027.02.1416:01'(공백 탈락) 허용
_P_CMP  = re.compile(r"(?<!\d)(20\d{2})(\d{2})(\d{2})(?!\d)")                          # 20270214 (점 탈락)
# 2자리 연도는 잡음과 혼동되기 쉬우므로 구분자 주변 공백 불허 + 같은 구분자 반복(역참조) 요구. '30.3- 7' 같은 조각 차단.
_P_YY   = re.compile(r"(?<!\d)(\d{2})([.\-/])(\d{1,2})\2(\d{1,2})(?!\d)")                # 21.04.24 / 25-12-02
_P_SP   = re.compile(r"(?<!\d)(\d{1,2})\s+(\d{1,2})\s+(\d{2,4})(?!\d)")                 # 30 12 23 (유럽식)
_P_DMY4 = re.compile(r"(?<!\d)(\d{1,2})[.\-/\s]+(\d{1,2})[.\-/\s]+(20\d{2})(?!\d)")      # 20/05/2026, '19 11.2027' — 연도가 맨 뒤면 일/월/년 (운영진 확정)
_P_CMP6 = re.compile(r"(?<!\d)(\d{2})(\d{2})(\d{2})(?!\d)")                                # 050926 — 'BEST BEFORE (DDMMYY)' 압축형 / 250626 (YYMMDD)
# 연도 없는 월.일 (예: 우유 '10.14 09:45'). 운영진 확정: NONE-10-14 로 출력.
# 앞뒤에 '.'+숫자가 붙으면(= 완전한 날짜의 일부) 제외. 2자리+2자리만 허용해 '4.9%' 류를 거른다.
_P_MMDD = re.compile(r"(?<![\d.])(\d{2})[.\-/](\d{2})(?![.\-/]?\d)")
# 연·월만 있는 날짜 (일 없음) → YYYY-MM-NONE (운영진 확정 필드별 NONE). 일본 賞味期限 '2027.7', 유럽 '12.2020'.
# 2자리 '25.11' 은 _P_MMDD 가 잡고 연도 범위(18~31)와 월(1~12)이 안 겹치는 것으로 구분한다.
_P_YYYYMM = re.compile(r"(?<!\d)(20\d{2})[.\-/\s]+(\d{1,2})(?![.\-/]?\d)")
_P_MMYYYY = re.compile(r"(?<![\d.])(\d{1,2})[.\-/\s]+(20\d{2})(?!\d)")


def _valid(y, m, d):
    return (y is None or YEAR_MIN <= y <= YEAR_MAX) and (1 <= m <= 12) and (d is None or 1 <= d <= 31)


def _fmt(y, m, d):
    """진단 출력용 'YYYY-MM-DD' (없는 필드는 NONE)."""
    return f"{y if y is not None else 'NONE'}-{f'{m:02d}' if m is not None else 'NONE'}-{f'{d:02d}' if d is not None else 'NONE'}"


def mask_long_digits(s):
    """품목보고번호(20130628332176)·바코드(8801052043838)처럼 앞자리가 날짜로 읽히는 긴 숫자열 제거."""
    return re.sub(r"\d{%d,}" % MASK_DIGITS_GE, " ", s)


def _parse_dates_one(text):
    """한 줄 텍스트 → [(y, m, d, kind)] 후보. kind 는 어떤 패턴으로 잡혔는지(진단용).
    신뢰도 순 3단계로 보고, 상위 단계에서 후보가 나오면 하위 단계는 보지 않는다:
      1) 구분자 있는 완전한 날짜 (2027.02.14 / 20270214 / 21.04.24)
      2) 공백 구분 (30 12 23) — 영양성분표 숫자열과 혼동되기 쉬워 1)이 없을 때만
      3) 연도 없는 월.일 (10.14) — 운영진 확정: NONE-MM-DD"""
    s = mask_long_digits(text)
    out = []
    for m in _P_YYYY.finditer(s):
        y, mo, d = int(m[1]), int(m[2]), int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "YYYY"))
    for m in _P_CMP.finditer(s):
        y, mo, d = int(m[1]), int(m[2]), int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "CMP"))
    for m in _P_DMY4.finditer(s):
        d, mo, y = int(m[1]), int(m[2]), int(m[3])
        if _valid(y, mo, d):
            out.append((y, mo, d, "DMY4"))
    for m in _P_YY.finditer(s):
        a, b, c = int(m[1]), int(m[3]), int(m[4])   # m[2] 는 구분자(역참조용)
        # 구분자(./-)가 있으면 한국 관행 YY.MM.DD 우선, 안 되면 DD.MM.YY.
        # 단, YY.MM.DD 로 읽은 연도가 YY_SOFT_MAX 를 넘고 DD.MM.YY 도 성립하면 후자 ('30/07/26' → 2030 이 아니라 2026-07-30).
        ymd_ok, dmy_ok = _valid(2000 + a, b, c), _valid(2000 + c, b, a)
        if ymd_ok and not (2000 + a > YY_SOFT_MAX and dmy_ok):
            out.append((2000 + a, b, c, "YY"))
        elif dmy_ok:
            out.append((2000 + c, b, a, "DDMMYY"))
    if not out:
        for m in _P_SP.finditer(s):
            a, b, c = int(m[1]), int(m[2]), int(m[3])
            cy = c if c >= 100 else 2000 + c
            # 공백 구분은 유럽식 DD MM YY 우선 (예: 이탈리아 제품 '30 12 23'), 안 되면 YY MM DD.
            # 2자리 연도는 YY_SOFT_MAX 이하만 — 기준선 실측: 이 단계 후보의 오답 대부분이 '2029-08-14' 류의 미래 연도 잡음.
            if _valid(cy, b, a) and cy <= max(YY_SOFT_MAX, c if c >= 100 else 0):
                out.append((cy, b, a, "SP_DMY"))
            elif c < 100 and _valid(2000 + a, b, c) and 2000 + a <= YY_SOFT_MAX:
                out.append((2000 + a, b, c, "SP_YMD"))
        for m in _P_CMP6.finditer(s):
            a, b, c = int(m[1]), int(m[2]), int(m[3])
            # 한국 관행 YYMMDD 우선, 안 되면 DDMMYY (수입 과자 'BEST BEFORE (DDMMYY) 050926'). LOT 번호 등은 연도 범위에서 대부분 탈락.
            if _valid(2000 + a, b, c) and 2000 + a <= YY_SOFT_MAX:
                out.append((2000 + a, b, c, "CMP6_YMD"))
            elif _valid(2000 + c, b, a) and 2000 + c <= YY_SOFT_MAX:
                out.append((2000 + c, b, a, "CMP6_DMY"))
    if not out:
        for m in _P_YYYYMM.finditer(s):
            y, mo = int(m[1]), int(m[2])
            if _valid(y, mo, None):
                out.append((y, mo, None, "YYYYMM"))
        for m in _P_MMYYYY.finditer(s):
            mo, y = int(m[1]), int(m[2])
            if _valid(y, mo, None):
                out.append((y, mo, None, "MMYYYY"))
        mmdd = [(int(m[1]), int(m[2])) for m in _P_MMDD.finditer(s)]
        # 'EXP:08/22 MFD:08/20' 처럼 NN/NN 이 둘 이상이고 뒤 숫자가 전부 연도 범위(17~31)면 월/연도 (수입품 관행) → 2022-08-NONE
        as_mmyy = len(mmdd) >= 2 and all(a <= 12 and _valid(2000 + b, a, None) for a, b in mmdd)
        for a, b in mmdd:
            if as_mmyy:
                out.append((2000 + b, a, None, "MMYY"))
            elif a >= 13 and _valid(2000 + a, b, None):   # '25.11' → 2025-11-NONE (a 가 13 이상이면 월일 수 없음)
                out.append((2000 + a, b, None, "YYMM"))
            elif _valid(None, a, b):                       # '10.14' → NONE-10-14 (한국 관행: 연도 없는 월.일)
                out.append((None, a, b, "MMDD"))
    seen, uniq = set(), []
    for t in out:
        if t[:3] not in seen:
            seen.add(t[:3])
            uniq.append(t)
    return uniq


def squash_digits(s):
    """도트 프린터 날짜는 '20 2 7 . 0 4.13' 처럼 검출·인식 단계에서 조각난다 (기준선 미인식 211건의 주원인).
    숫자·구분자 사이의 공백을 없애 '2027.04.13' 으로 만든다."""
    return re.sub(r"(?<=[\d.\-/])\s+(?=[\d.\-/])", "", s)


def parse_dates(text):
    """원문과 공백 제거본을 모두 파싱해 합친다.
    공백 구분 날짜('30 12 23')는 원문에서, 조각난 날짜('20 2 7 . 0 4.13')는 제거본에서 잡힌다. 잡음은 대부분 9자리 이상으로 붙어 마스킹된다."""
    out = _parse_dates_one(text)
    sq = squash_digits(text)
    if sq != text:
        seen = {t[:3] for t in out}
        out += [t for t in _parse_dates_one(sq) if t[:3] not in seen]
    return out


# 후보 선택 시 패턴 신뢰 등급. 기준선 실측 정밀도(완전일치): YYYY 80% · DMY4 78% · YY 74% · CMP 83%  vs  SP_DMY 12% · CMP6 0~13% · MMDD 29%(부분)
# → 같은 이미지에 1등급 후보가 있으면 2·3등급은 보지 않는다. (줄 단위 3단계와 별개로 이미지 단위 적용)
KIND_TIER = {"YYYY": 1, "CMP": 1, "DMY4": 1, "YY": 1, "DDMMYY": 1, "MON": 1,
             "SP_DMY": 2, "SP_YMD": 2, "CMP6_YMD": 2, "CMP6_DMY": 2,
             "YYYYMM": 3, "MMYYYY": 3, "YYMM": 3, "MMYY": 3, "MMDD": 3}


_MON_RE = re.compile(r"(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)")
MONTHS  = {"JAN": 1, "FEB": 2, "MAR": 3, "APR": 4, "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8, "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12}


def parse_mon(text):
    """영문 월 이름이 있는 텍스트 → 후보. 'NOV 29 2021' / '29 NOV 21' / 'NOV 2021'(일 없음) / 'SEP 5 26'.
    월 이름 주변 창에서 4자리(20xx)는 연도, 1~2자리는 일. 2자리가 둘이면 앞이 일·뒤가 연도."""
    s = mask_long_digits(text.upper())
    out = []
    for mm in _MON_RE.finditer(s):
        mo = MONTHS[mm.group(1)]
        win = s[max(0, mm.start() - 8): mm.end() + 12]
        nums = []
        for t in re.findall(r"\d+", win):                 # 붙어 찍힌 'AUG292020' → 29 + 2020, 'NOV2921' → 29 + 21
            if len(t) == 6 and t[2:4] == "20":
                nums += [t[:2], t[2:]]
            elif len(t) == 4 and not t.startswith("20"):
                nums += [t[:2], t[2:]]
            else:
                nums.append(t)
        n4 = [int(t) for t in nums if len(t) == 4 and t.startswith("20")]
        n2 = [int(t) for t in nums if len(t) <= 2]
        if n4:
            y, d = n4[0], (n2[0] if n2 else None)
        elif n2:
            d = n2[0]
            yy = [t for t in n2[1:] if YEAR_MIN - 2000 <= t <= YEAR_MAX - 2000]   # 일 뒤의 2자리 중 연도 범위인 첫 값 ('SEP 5 26 LOT A3' 의 3 배제)
            y = 2000 + yy[0] if yy else None
        else:
            continue
        if _valid(y, mo, d):
            out.append((y, mo, d, "MON"))
    return out


# 빠른 자가검증 (실행 시 assert 통과해야 함)
assert parse_mon("BBE NOV 29 2021")[0][:3]             == (2021, 11, 29)
assert parse_mon("26 MAR 21")[0][:3]                   == (2021, 3, 26)
assert parse_mon("NOV 2021")[0][:3]                    == (2021, 11, None)
assert parse_mon("MAY CONTAIN NUTS 1234567890")        == []
assert parse_dates("소비기한 2027.06.26 까지")[0][:3] == (2027, 6, 26)
assert parse_dates("품목보고번호 20130628332176")     == []            # 14자리 마스킹
assert parse_dates("21.04.24 까지 10:47")[0][:3]       == (2021, 4, 24)
assert parse_dates("30 12 23")[0][:3]                  == (2023, 12, 30) # DD MM YY
assert parse_dates("2020.12.28/16:35 2021.12.27(3)")   and len(parse_dates("2020.12.28/16:35 2021.12.27(3)")) == 2
assert parse_dates("NEL 1857 CANNELLA")                == []
assert parse_dates("10.14 09:45 PA")[0][:3]            == (None, 10, 14) # 연도 없음 → NONE-10-14
assert all(t[0] is not None for t in parse_dates("2020.12.28"))          # 완전한 날짜에서 월.일 오탐 금지
assert parse_dates("4.9 2450 100 130")                 == []
assert parse_dates("BBD: 20/05/2026 YYT Q5")[0][:3]    == (2026, 5, 20)  # 연도 뒤 → 일/월/년 (운영진 확정)
assert parse_dates("050926 213")[0][:3]                == (2026, 9, 5)   # DDMMYY 압축형: 05일 09월 26년 (운영진 확정)
assert parse_dates("LOT NO 543123")                    == []             # 6자리 LOT 은 연도 범위에서 탈락
assert parse_dates("5069351 930117 113809")            == []
assert parse_dates("2027.7 /WR/+1")[0][:3]             == (2027, 7, None)  # 연·월만 → 2027-07-NONE (일본 賞味期限)
assert parse_dates("12.2020 -13:28")[0][:3]            == (2020, 12, None) # 유럽식 월.연도
assert parse_dates("25.11")[0][:3]                     == (2025, 11, None) # 2자리 연·월
assert parse_dates("2025.06.18 09:16")[0][:3]          == (2025, 6, 18)    # 완전한 날짜에서 연·월 오탐 금지
assert (2027, 4, 13) in [t[:3] for t in parse_dates("20 2 7 . 0 4.13")]        # 조각난 도트 프린터 날짜
assert (2026, 7, 30) in [t[:3] for t in parse_dates("2 026 . 0 7 . 3 0 77 7/")]
assert (2025, 10, 9) in [t[:3] for t in parse_dates("42025.10.0971741-5834")]   # 앞뒤 잡음 숫자
assert (2027, 11, 19) in [t[:3] for t in parse_dates("19 11.2027 012  7428")]   # 공백+점 혼합 일/월/년
assert not any(t[0] and t[0] > 2027 and t[3].startswith(("SP", "CMP6")) for t in parse_dates("28 08 30"))  # 2자리 연도 상한
print("parse_dates 자가검증 통과")

In [ ]:
reader = easyocr.Reader(
    ["en"], gpu=USE_GPU,
    model_storage_directory=WEIGHTS_DIR,
    download_enabled=False,   # 오프라인 강제: 가중치가 없으면 여기서 즉시 실패해야 한다
)
print("EasyOCR 로드 완료 (offline)")

In [ ]:
def group_lines(results, y_tol=0.6):
    """readtext 결과를 같은 줄끼리 묶어 [(text, (x0,y0,x1,y1), conf)] 반환.
    '30 12 23' 처럼 공백으로 갈라진 날짜, '2020.12.28/16:35' 처럼 붙은 시간을 한 줄로 본다."""
    items = []
    for box, text, conf in results:
        xs = [p[0] for p in box]
        ys = [p[1] for p in box]
        items.append({"x0": min(xs), "y0": min(ys), "x1": max(xs), "y1": max(ys),
                      "text": text, "conf": float(conf)})
    items.sort(key=lambda t: ((t["y0"] + t["y1"]) / 2, t["x0"]))
    lines = []
    for it in items:
        cy, h = (it["y0"] + it["y1"]) / 2, it["y1"] - it["y0"]
        for ln in lines:
            lcy, lh = (ln["y0"] + ln["y1"]) / 2, ln["y1"] - ln["y0"]
            if abs(cy - lcy) <= max(h, lh) * y_tol:
                ln["items"].append(it)
                ln["x0"], ln["x1"] = min(ln["x0"], it["x0"]), max(ln["x1"], it["x1"])
                ln["y0"], ln["y1"] = min(ln["y0"], it["y0"]), max(ln["y1"], it["y1"])
                break
        else:
            lines.append({"x0": it["x0"], "y0": it["y0"], "x1": it["x1"], "y1": it["y1"], "items": [it]})
    for ln in lines:
        ln["items"].sort(key=lambda t: t["x0"])
        ln["text"] = " ".join(t["text"] for t in ln["items"])
        ln["conf"] = float(np.mean([t["conf"] for t in ln["items"]]))
    return lines   # 각 원소: {x0,y0,x1,y1,text,conf,items:[단어 박스…]}


def ocr_prioritized(small):
    """검출 1회 → 박스를 글자 높이 내림차순으로 인식(조기 종료·상한 적용). readtext 와 같은 [(box, text, conf)] 반환.
    기울어진 폴리곤(free_list)은 축 정렬 bbox 로 바꿔 같이 다룬다."""
    hl, fl = reader.detect(small, min_size=10)
    boxes = [tuple(map(int, b)) for b in hl[0]]                         # (x0, x1, y0, y1)
    for poly in fl[0]:
        xs, ys = [p[0] for p in poly], [p[1] for p in poly]
        boxes.append((int(min(xs)), int(max(xs)), int(min(ys)), int(max(ys))))
    boxes.sort(key=lambda b: -(b[3] - b[2]))                              # 큰 글자 먼저
    grey = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    out, stop_h, seen_datey = [], None, False
    cap_soft, cap_hard = (10, 10) if FAST_LEVEL >= 2 else (MAX_BOXES, MAX_BOXES_HARD)   # 최소 단계에선 큰 글자 10개만
    for i, (x0, x1, y0, y1) in enumerate(boxes[:cap_hard]):
        h = y1 - y0
        if stop_h is not None and h < stop_h:
            break
        if i >= cap_soft and seen_datey:        # 소프트 상한: 날짜 비슷한 텍스트를 이미 봤으면 여기서 멈춤
            break
        for _, text, conf in reader.recognize(grey, [[x0, x1, y0, y1]], [], allowlist=ALLOW_P1, reformat=False):
            out.append(([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], text, float(conf)))
            if re.search(r"20\d{2}|\d{2}[./-]\d{2}", text):
                seen_datey = True
            # 조기 종료는 '완전한 1등급 날짜'를 읽었을 때만. 연·월만 읽힌 조각('2026.07')에 멈추면 일('03') 박스를 못 읽는다 (after 실측 퇴행 4건).
            if stop_h is None and any(p[0] is not None and p[2] is not None and KIND_TIER.get(p[3], 3) == 1 for p in parse_dates(text)):
                stop_h = h * EARLY_STOP_RATIO
    return out, grey, boxes


def recognize_boxes(grey, boxes, allowlist):
    """검출된 박스들을 주어진 allowlist 로 인식. readtext 와 같은 [(box, text, conf)] 반환."""
    out = []
    for (x0, x1, y0, y1) in boxes:
        for _, text, conf in reader.recognize(grey, [[x0, x1, y0, y1]], [], allowlist=allowlist, reformat=False):
            out.append(([[x0, y0], [x1, y0], [x1, y1], [x0, y1]], text, float(conf)))
    return out


def _cands_from_line(ln, s, src, parser):
    """한 줄(축소본 좌표) → 후보 dict 목록. bbox·items 는 원본 좌표로 환산."""
    return [{"y": y, "m": m, "d": d, "kind": kind, "conf": ln["conf"], "src": src, "text": ln["text"],
             "bbox":  (ln["x0"] / s, ln["y0"] / s, ln["x1"] / s, ln["y1"] / s),
             "items": [(it["x0"] / s, it["y0"] / s, it["x1"] / s, it["y1"] / s) for it in ln["items"]]}
            for (y, m, d, kind) in parser(ln["text"])]


def pass1(img):
    """해상도 사다리를 오르며 축소본 전체 OCR. 후보가 나오는 첫 단계에서 멈춘다.
    반환: (후보 리스트, [(해상도, OCR 라인들)] 진단용). bbox 는 원본 좌표로 환산해 둔다."""
    seen = []
    ladder = PASS1_LADDER if FAST_LEVEL == 0 else PASS1_LADDER[:1]   # 1·2 단계: 640 만 (480 으로 줄이면 인식 붕괴 — 실측 6%)
    for idx, L in enumerate(ladder):
        small, s = resize_long(img, L)
        res, grey, boxes = ocr_prioritized(small)
        lines = group_lines(res)
        seen.append((L, lines))
        cands = [c for ln in lines for c in _cands_from_line(ln, s, f"p1@{L}", parse_dates)]
        if cands:
            return cands, seen
        last = idx == len(ladder) - 1 or s >= 1.0     # 더 올릴 단계가 없음 (마지막 단계이거나 이미 원본 크기)
        if last and FAST_LEVEL == 0:
            # 영문 월 이름 폴백: 사다리를 다 쓴 뒤 한 번만 (~2초). 매 단계마다 돌렸더니 장당 3.9→6.2s 로 악화됐음 (after 실측).
            mon_lines = group_lines(recognize_boxes(grey, boxes[:MON_FALLBACK_BOXES], ALLOW_MON))
            cands = [c for ln in mon_lines for c in _cands_from_line(ln, s, f"mon@{L}", parse_mon)]
            if cands:
                seen.append((f"mon@{L}", mon_lines))
                return cands, seen
            break
    return [], seen


def pass2(img, cands):
    """후보 영역만 원본에서 크롭해 **인식기만** 다시 돌린다 (검출기는 장당 1회로 제한 — CPU 비용의 대부분이 검출기).
    크롭 전체를 한 줄로 보고 recognize() 하므로 검출 없이 수십 ms 에 끝난다.
    재인식이 파싱되면 그 결과로 대체, 아니면 1패스 값 유지.
    크롭에 이웃 줄(제조일자 등)이 같이 들어와 날짜가 여러 개 나오면 전부 후보로 올린다 → 선택 규칙이 처리."""
    out = []
    for c in cands:
        if c["src"].startswith("mon"):        # 영문 월 후보는 숫자 allowlist 재인식으로 월을 잃으므로 2패스 생략
            out.append(c)
            continue
        texts, confs = [], []
        for ib in c["items"]:                 # 줄 전체가 아니라 검출된 단어 박스 단위로 재인식 → 이웃 줄이 섞여 들어오는 것 방지
            crop = crop_with_margin(img, ib, PASS2_MARGIN)
            if crop is None:
                continue
            res = reader.recognize(crop, allowlist=ALLOW_P2)   # horizontal_list 생략 → 크롭 전체를 한 박스로 인식
            texts += [t for _, t, _ in res]
            confs += [float(cf) for _, _, cf in res]
        text = " ".join(texts)
        conf = float(np.mean(confs)) if confs else 0.0
        parsed = parse_dates(text)
        # 2패스는 '교체'가 아니라 '확인·보완'만 한다. 기준선 실측: 2패스가 덮어쓴 답의 완전일치 61.7% vs 1패스 유지 72.3%.
        out.append(c)                                             # 1패스 값은 항상 유지
        for (y, m, d, kind) in parsed:
            if (y, m, d) == (c["y"], c["m"], c["d"]):             # 확인: 같은 값을 다시 읽었으면 신뢰도만 올린다
                c["conf"] = max(c["conf"], conf)
                c["src"] = c["src"] + "+p2"
            elif (y is not None and d is not None and m == c["m"] and (c["y"] is None or c["d"] is None)
                  and (c["d"] is None or d == c["d"]) and (c["y"] is None or y == c["y"])):
                out.append({**c, "y": y, "m": m, "d": d, "kind": kind, "conf": conf, "src": "p2", "text": text})   # 보완: 놓친 연도/일을 채움
    return out


def select_date(cands):
    """운영진 확정 규칙: 날짜가 여럿이면 가장 늦은 것이 소비기한. 동률은 신뢰도.
    단 '가장 늦은 날짜' 규칙은 미래 연도 잡음('2029-08-14')에 취약하므로, 먼저 패턴 등급(KIND_TIER)이 가장 높은 후보들로 좁힌다."""
    if not cands:
        return None
    best_tier = min(KIND_TIER.get(c["kind"], 3) for c in cands)
    pool = [c for c in cands if KIND_TIER.get(c["kind"], 3) == best_tier]
    # 연도 없는 후보(y=None)는 완전한 날짜보다 항상 뒤로 보낸다
    return max(pool, key=lambda c: ((c["y"] or 0, c["m"], c["d"] or 0), c["conf"]))

In [ ]:
all_files   = sorted(glob.glob(os.path.join(INPUT_DIR, "*.*")))
image_files = [p for p in all_files if os.path.splitext(p)[1].lower() in IMG_EXT]
skipped     = [os.path.basename(p) for p in all_files if p not in set(image_files)]
if skipped:
    print(f"[INFO] 이미지 확장자가 아니어서 건너뜀: {skipped[:10]}{' ...' if len(skipped) > 10 else ''}")
print(f"입력 {len(image_files)}장  ({INPUT_DIR})")

rows, kinds, dbg_rows = [], Counter(), []
secs, level_since = [], 0          # 시간 예산 가드용: 장별 소요, 마지막으로 단계를 올린 시점
n_fallback = n_p2 = n_err = 0
t_all = time.time()

for i, path in enumerate(image_files, 1):
    img_id = os.path.splitext(os.path.basename(path))[0]   # 확장자 제외 파일명 그대로. zero-pad 가정 금지.
    best, cands, p1_lines, t0 = None, [], [], time.time()
    try:
        if FAST_LEVEL < 3:                       # 3 (비상) 이면 처리 없이 NONE — CSV 생성이 최우선
            img = load_image(path)
            cands, p1_lines = pass1(img)
            if cands and FAST_LEVEL < 2:         # 2 이상이면 2패스 생략
                cands = pass2(img, cands)
            best = select_date(cands)
    except Exception as e:
        n_err += 1
        print(f"[WARN] {img_id}: {type(e).__name__}: {e}")
    if DEBUG:
        picked = f"{_fmt(best['y'], best['m'], best['d'])} ({best['kind']}/{best['src']})" if best else "NONE"
        print(f"  {img_id}: {time.time() - t0:.1f}s → {picked}")
        for c in cands:
            print(f"      cand {_fmt(c['y'], c['m'], c['d'])} {c['kind']:7s} {c['src']:7s} conf={c['conf']:.2f}  text='{c['text'][:80]}'")
        if not cands:
            for L, lines in p1_lines:   # 후보가 없을 때 OCR 이 실제로 뭘 읽었는지 (숫자 2개 이상 포함 라인만)
                digs = [(ln["text"][:40], round(ln["conf"], 2)) for ln in lines if re.search(r"\d{2}", ln["text"])]
                print(f"      p1@{L} 숫자 라인 {len(digs)}개: {digs[:10]}")

    if best is None:
        n_fallback += 1
        y, m, d = PRIOR_DATE if NONE_POLICY == "prior" else (None, None, None)
    else:
        y, m, d = best["y"], best["m"], best["d"]
        kinds[best["kind"]] += 1
        if best["src"] == "p2":
            n_p2 += 1

    # 각 필드는 독립적으로 NONE 가능. final_date 는 세 필드를 '-' 로 이은 것 (예: NONE-10-14, 운영진 확정 포맷).
    ys = f"{y:04d}" if y is not None else "NONE"
    ms = f"{m:02d}" if m is not None else "NONE"
    ds = f"{d:02d}" if d is not None else "NONE"
    fd = "NONE" if (y is None and m is None and d is None) else f"{ys}-{ms}-{ds}"
    rows.append({"image_id": img_id, "year": ys, "month": ms, "day": ds, "final_date": fd})
    if DEBUG_CSV:
        dbg_rows.append({"image_id": img_id, "final_date": fd, "conf": round(best["conf"], 3) if best else "",
                         "kind": best["kind"] if best else "", "src": best["src"] if best else "",
                         "n_cands": len(cands), "sec": round(time.time() - t0, 2),
                         "cands": "|".join(f"{c['y']}-{c['m']}-{c['d']}:{c['kind']}:{c['src']}:{c['conf']:.2f}" for c in cands)})

    # 시간 예산 가드: 최근 20장 속도로 남은 소요를 예측해 예산 초과가 예상되면 경량 단계를 올린다 (단계를 올린 뒤 20장은 새 속도를 재고 다시 판단).
    secs.append(time.time() - t0)
    if FAST_LEVEL < 3 and i >= 20 and i - level_since >= 20:
        recent = sum(secs[-20:]) / 20
        projected = (time.time() - t_all) + recent * (len(image_files) - i)
        if projected > TIME_BUDGET_S:
            FAST_LEVEL += 1
            level_since = i
            print(f"[GUARD] {i}장 시점 예상 총 소요 {projected:.0f}s > 예산 {TIME_BUDGET_S}s → 경량 {FAST_LEVEL}단계 (최근 20장 평균 {recent:.2f}s/장)")
    if i % 50 == 0 or i == len(image_files):
        el = time.time() - t_all
        print(f"[{i}/{len(image_files)}] {el:.0f}s 경과 · 장당 {el / i:.2f}s · 후보없음 {n_fallback} · 오류 {n_err}")

In [ ]:
df = pd.DataFrame(rows, columns=["image_id", "year", "month", "day", "final_date"])
df.to_csv(OUTPUT_PATH, index=False)
if DEBUG_CSV:
    pd.DataFrame(dbg_rows).to_csv(DEBUG_CSV, index=False)

total = time.time() - t_all
print(f"Saved {len(df)} rows → {OUTPUT_PATH}")
print(f"총 {total:.1f}s · 장당 {total / max(1, len(df)):.2f}s · 후보없음 {n_fallback} · 2패스 채택 {n_p2} · 오류 {n_err}")
print("패턴 분포:", dict(kinds))
df.head(10)